# NetFlow Traffic Classification Lab

This notebook builds a supervised ML classifier for **network flow** data (NetFlow/IPFIX-style records).

## Objectives
- Treat each row as a *single network flow record* and learn a model that can **predict the flow’s class**.
- Build an end-to-end pipeline (preprocessing → model) that can be reused on new/unseen flows.
- Compare a simple linear baseline (Logistic Regression) vs a non-linear baseline (Random Forest).
- Evaluate how well the model generalizes using a held-out test set (precision/recall/F1 per class).
- Identify which fields are most useful for discrimination (permutation importance).

## What is a “flow” in this lab?
A **flow** is an *aggregated summary of packets*, typically defined by a 5‑tuple and a time window:
- Source IP, Destination IP
- Source port, Destination port
- L4 protocol (TCP/UDP/ICMP, …)


An exporter (router/switch/agent) aggregates packets into a flow record and outputs counters such as duration, packets, bytes, flags, etc. In this dataset, **one row = one flow record** (not raw packets).

## Overall strategy to distinguish flows
We assume different traffic types leave different “signatures” in flow features, for example:
- **Ports / protocol** (DNS/53, HTTPS/443, SSH/22, …)
- **Volume & timing** (bytes, packets, duration, packets/sec, bytes/sec)
- **Directionality / asymmetry** (client→server vs server→client patterns)
- **Categorical context** (e.g., protocol name, interface, exporter metadata if present)

The notebook’s approach is:
1. Split columns into numeric vs categorical.
2. Handle missing values (median for numeric, most-frequent for categorical).
3. Scale numeric features and one-hot encode categorical features.
4. Train a classifier and evaluate on a separate test split.

## What do we classify at the end?
The target is the `Label` column (`y = df['Label']`). The model learns to predict `Label` for each flow using all other columns as input features.
Depending on your dataset, `Label` can represent application/service type, traffic category, or benign vs specific attack types.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance

In [ ]:
# Load dataset
df = pd.read_csv('flows.csv') # flows.csv is the NetFlow dataset with a 'Label' column for classification
df = df.dropna(subset=['Label'])  # Drop rows with missing labels

In [ ]:
# Show a few examples
df.head(10)

## Data Preparation

Before training a model, we need to turn the raw flow table into a clean, fully numeric feature matrix that the algorithms can learn from. 

The first step is to identify which columns are **numeric** (e.g., bytes, packets, duration) and which are **categorical** (e.g., protocol names, flags represented as text, interface names). 

Then we check for **missing values**: for numeric columns we typically replace missing entries with a robust statistic such as the **median**, and for categorical columns we replace missing entries with the **most frequent** category, so that no rows are dropped unnecessarily. 

Next, because models cannot consume raw text categories directly, we convert categorical columns using **one-hot encoding**, which creates binary indicator columns for each category and also safely handles categories that appear later in test data. 

For **Logistic Regression**, feature scaling is especially important because it is a linear model trained by gradient-based optimization: if one feature is measured in thousands (bytes) and another in small units (duration), the large-scale feature can dominate unless we apply **standardization** (zero mean, unit variance). 

For **Random Forest**, scaling is not strictly required because trees split on thresholds and are not sensitive to feature magnitude in the same way, but we still need the data to be numeric and free of missing values; therefore imputation and encoding remain essential. 

The overall goal of preprocessing is to produce consistent training and test inputs with the same columns and clean values, so both models can be trained and evaluated fairly on the same transformed representation of the flows.

In [ ]:
features = [col for col in df.columns if col != 'Label'] # Get all columns except 'Label' as features
X = df[features] # Feature matrix
y = df['Label'] # Target variable

X_train, X_test, y_train, y_test = train_test_split( # Split the data into training and testing sets
    X, y, test_size=0.25, stratify=y, random_state=42
)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist() # Identify numeric features 
categorical_features = X.select_dtypes(include=['object']).columns.tolist() # Identify categorical features

numeric_transformer = Pipeline([ #  Pipeline for numeric features: impute missing values with median and scale
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([ # Pipeline for categorical features: impute missing values with most frequent and one-hot encode
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer([ # Combine numeric and categorical transformers
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## Logistic Regression

This section trains and evaluates a Logistic Regression classifier using the same preprocessing `pipeline` defined earlier. The Pipeline is important because it guarantees that the exact same transformations (imputation, scaling of numeric features, and one‑hot encoding of categorical features) are applied consistently during both training and testing, which avoids data leakage and prevents mismatched feature columns. 

The model itself is `LogisticRegression(max_iter=2000, class_weight='balanced')`: `max_iter=2000` increases the number of optimization steps to help the solver converge, and `class_weight='balanced'` automatically gives more weight to minority classes so the classifier does not simply favor the most frequent label. 

`logreg.fit(X_train, y_train)` learns the model parameters from the training flows, `logreg.predict(X_test)` produces predicted labels for unseen test flows, and `classification_report(y_test, pred)` prints per-class precision, recall, and F1-score (plus averages), which provides a clear view of how well the model distinguishes each traffic class.

In [10]:
# Logistic Regression
logreg = Pipeline([ #
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

logreg.fit(X_train, y_train)
pred = logreg.predict(X_test)
print('Logistic Regression Results')
print(classification_report(y_test, pred))

Logistic Regression Results
              precision    recall  f1-score   support

        BULK       0.99      0.98      0.98      1857
       VIDEO       0.99      0.99      0.99      2484
        VOIP       0.98      0.99      0.98      1874
         VPN       0.96      0.95      0.95      1921
         WEB       0.98      1.00      0.99      4364

    accuracy                           0.98     12500
   macro avg       0.98      0.98      0.98     12500
weighted avg       0.98      0.98      0.98     12500



In `sklearn.metrics.classification_report`, each row corresponds to a class (a value of your `Label`). 

The columns mean:

- **precision**: “When the model predicts this class, how often is it correct?”

$$\text{precision}=\frac{TP}{TP+FP}$$

High precision ⇒ few false positives for that class.

- **recall**: “Of all the true items of this class, how many did the model find?”

$$
  \text{recall}=\frac{TP}{TP+FN}
$$

High recall ⇒ few false negatives for that class.

- **f1-score**: single number that balances precision and recall (harmonic mean).

$$
  F_1=\frac{2\cdot(\text{precision}\cdot\text{recall})}{\text{precision}+\text{recall}}
$$

It drops if either precision or recall is low.

- **support**: how many true samples of that class are in the test set (i.e., the count of that label in `y_test`).

Support matters because metrics for very small-support classes are less stable, and it drives the difference between “macro” and “weighted” averages.

Let's understand the results for the `BULK` traffic.

```
        precision    recall  f1-score   support
BULK       0.99      0.98      0.98      1857
```

Interpretation:

- **support = 1857** means there are **1,857** true `BULK` flows in the test set.
- **recall = 0.98** means the model correctly finds about **98%** of true `BULK` flows (few false negatives).
- **precision = 0.99** means that among flows predicted as `BULK`, about **99%** are truly `BULK` (few false positives).
- **f1 = 0.98** summarizes the precision/recall balance for that class.

Approximate counts (because the report is rounded):

- $FN \approx (1-0.98)\times1857 \approx 37$ `BULK` flows missed.
- If $TP \approx 0.98\times1857 \approx 1820$, then $FP$ implied by precision is roughly:

  $$
  FP \approx TP\left(\frac{1}{0.99}-1\right) \approx 18
  $$

So roughly ~18 non-`BULK` flows were incorrectly predicted as `BULK`.

Let's understand the results for the `VPN` traffic:

```
      precision    recall  f1-score   support
VPN       0.96      0.95      0.95      1921
```

it is often slightly worse because:

- **VPN is heterogeneous** (many behaviors/protocols under one label).
- **Flow-level features overlap** with other encrypted traffic (often similar to TLS/HTTPS).
- **Missing discriminative signals** at flow level (no TLS SNI, no cert metadata, no DPI signatures).
- **Label noise** can be higher for “VPN” in real datasets.

To conclude, VPN may not form a single clean cluster in flow-feature space.



Let's understand the summary values:



```
            precision    recall  f1-score   support
accuracy                            0.98     12500
macro avg       0.98      0.98      0.98     12500
weighted avg    0.98      0.98      0.98     12500
```

- **accuracy = 0.98** on **12,500** test flows means **98%** were predicted correctly.

  $$
  \text{accuracy}=\frac{\#\text{correct}}{\#\text{total}}
  $$

- **macro avg**: compute metric per class, then average them **equally** (rare classes count as much as frequent ones).
- **weighted avg**: average metrics **weighted by support** (frequent classes count more).

If macro and weighted averages are close, performance is likely strong across both common and rarer classes (or class imbalance is low).

## Random Forest

This section trains and evaluates a Random Forest classifier using the exact same preprocessing pipeline as before. Wrapping everything in a Pipeline is important because it ensures that missing-value imputation and one‑hot encoding are applied consistently at both training time and test time, and it prevents mistakes like training on differently transformed features than the ones used for evaluation. 

The model is `RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample')`: using `300` trees typically improves stability and performance compared to a very small forest, and `balanced_subsample` adjusts class weights on each bootstrap sample so minority classes are not ignored during training. 

The call `rf.fit(X_train, y_train)` learns an ensemble of decision trees from the training flows, `rf.predict(X_test)` outputs predicted labels for unseen test flows, and `classification_report(y_test, pred_rf)` summarizes performance with precision/recall/F1 per class, letting you see which traffic types the forest separates well and which ones it still confuses.

In [11]:
# Random Forest
rf = Pipeline([ # Pipeline for Random Forest: preprocess the data and then fit a Random Forest model
    ('preprocess', preprocess),
    ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample'))
])

rf.fit(X_train, y_train) # Fit the Random Forest model on the training data
pred_rf = rf.predict(X_test) # Predict the labels for the test set using the Random Forest model
print('Random Forest Results')
print(classification_report(y_test, pred_rf))



Random Forest Results
              precision    recall  f1-score   support

        BULK       0.99      1.00      0.99      1857
       VIDEO       1.00      0.99      0.99      2484
        VOIP       1.00      1.00      1.00      1874
         VPN       0.99      0.98      0.99      1921
         WEB       1.00      1.00      1.00      4364

    accuracy                           0.99     12500
   macro avg       0.99      0.99      0.99     12500
weighted avg       0.99      0.99      0.99     12500



Compared to the earlier Logistic Regression run (where reported overall metrics around 0.98, with examples like BULK at F1 ≈ 0.98 and VPN at F1 ≈ 0.95), the Random Forest is clearly stronger and more consistent across classes:

- Overall

Accuracy: Random Forest 0.99 vs Logistic Regression ~0.98 (so roughly half as many mistakes on the 12,500 test flows).
Macro / weighted avg: Random Forest 0.99 / 0.99 vs Logistic Regression ~0.98 / ~0.98 → the gain is not just on the majority class; performance is high across classes.

- Per-class highlights: 

    - VPN (the class we noticed as “worse” before):  
    Logistic Regression example gave precision 0.96, recall 0.95 and F1 0.95 when Random Forest is now: precision 0.99, recall 0.98 and F1 0.99
    The forest produces far fewer confusions between VPN and other “similar-looking” encrypted traffic.  

    - BULK:  
    Logistic Regression example gave: precision 0.99, recall 0.98, F1 0.98 when Random Forest is now: precision 0.99, recall 1.00, F1 0.99
    Almost no BULK flows are missed anymore (recall went up).

    - VIDEO / VOIP / WEB:  
    Random Forest is essentially near-perfect (F1 ≈ 0.99–1.00), which suggests these classes have strong, separable patterns in the flow features.

- Why Random Forest often beats Logistic Regression here: 

Logistic Regression is linear: it can struggle when class boundaries depend on non-linear thresholds and feature interactions (e.g., “bytes and duration and packet rate together”).   
Random Forest captures non-linear rules (lots of “if feature > threshold then…” splits) and interactions naturally, which is often a better match for flow-statistics data.

This section computes permutation feature importance to understand which flow fields the trained Random Forest relies on most.   

The idea is simple: if a feature is important, then randomly shuffling its values in the test set should hurt the model’s performance, because the relationship between that feature and the label is broken while everything else stays the same.   

permutation_importance(rf, X_test, y_test, n_repeats=5) measures that effect by permuting each feature several times (5 repeats) and averaging the resulting drop in score to make the estimate more stable. 

The line importances = pd.Series(result.importances_mean, index=features) turns the mean importance values into a labeled series, and sort_values(...).head(10) prints the top 10 features with the largest average impact on performance. Interpreting the output: higher values mean “shuffling this column hurts the model more,” so that column is more influential for the classifier’s decisions.

In [12]:


# Permutation importance to identify which features are most important for the Random Forest model
result = permutation_importance(rf, X_test, y_test, n_repeats=5)
importances = pd.Series(result.importances_mean, index=features)
print(importances.sort_values(ascending=False).head(10))

TotBwdPkts        0.358352
DstPort           0.256960
TotFwdPkts        0.234736
TotLenFwdBytes    0.188832
TotLenBwdBytes    0.156080
FlowPktsPerS      0.012608
FlowBytsPerS      0.009712
ACKFlagCnt        0.000240
Protocol          0.000112
RSTFlagCnt        0.000064
dtype: float64


The permutation importance results show that the Random Forest mainly distinguishes traffic classes using a few very intuitive flow statistics: packet counts and byte volumes, especially in the backward (server→client) direction, plus the destination port. Features like TotBwdPkts, TotFwdPkts, TotLenFwdBytes, and TotLenBwdBytes capture the “shape” of the conversation (how much data and how many packets move in each direction), which differs a lot between applications such as web browsing, video streaming, VoIP, bulk transfers, and VPN. 

In contrast, rate features (FlowPktsPerS, FlowBytsPerS) add only a small amount of extra information here, and TCP flags / protocol contribute almost nothing, meaning that for this dataset the traffic classes are explained much more by volume and directionality patterns than by low-level protocol flag behavior.